## 1. Import of the libraries + setup the path

In [1]:
from pathlib import Path
import os
import xml.etree.ElementTree as ET
from PIL import Image
from tqdm import tqdm

# Paths adjusted for notebooks folder structure
PROJECT_ROOT = Path.cwd().parent  # one level up from notebooks/

INPUT_PATH = PROJECT_ROOT / "data"
INPUT_PATH = INPUT_PATH.resolve()
OUTPUT_PATH = PROJECT_ROOT / "data_processed" 
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_PATH.resolve()

# Annotations (nested folders exist in this dataset layout)
TRAIN_ANNOTATIONS_DIR = INPUT_PATH / "DETRAC-Train-Annotations-XML" / "DETRAC-Train-Annotations-XML"
TRAIN_ANNOTATIONS_DIR = TRAIN_ANNOTATIONS_DIR.resolve()
TEST_ANNOTATIONS_DIR  = INPUT_PATH / "DETRAC-Test-Annotations-XML" / "DETRAC-Test-Annotations-XML"
TEST_ANNOTATIONS_DIR = TEST_ANNOTATIONS_DIR.resolve()

# Images (nested folders exist in this dataset layout)
ALL_IMAGES_DIR = INPUT_PATH / "DETRAC-Images" / "DETRAC-Images"
ALL_IMAGES_DIR = ALL_IMAGES_DIR.resolve()

# Optional: quick sanity checks
assert TRAIN_ANNOTATIONS_DIR.is_dir(), f"Missing: {TRAIN_ANNOTATIONS_DIR}"
assert TEST_ANNOTATIONS_DIR.is_dir(), f"Missing: {TEST_ANNOTATIONS_DIR}"
assert ALL_IMAGES_DIR.is_dir(), f"Missing: {ALL_IMAGES_DIR}"

## 2. Train/Test split

In [2]:
import shutil
from glob import glob
from pathlib import Path
from tqdm import tqdm

# --- Checks (Path objects) ---
assert TRAIN_ANNOTATIONS_DIR.is_dir(), f"TRAIN_ANNOTATIONS_DIR not found: {TRAIN_ANNOTATIONS_DIR}"
assert TEST_ANNOTATIONS_DIR.is_dir(), f"TEST_ANNOTATIONS_DIR not found: {TEST_ANNOTATIONS_DIR}"
assert ALL_IMAGES_DIR.is_dir(), f"ALL_IMAGES_DIR not found: {ALL_IMAGES_DIR}"
assert OUTPUT_PATH.is_dir(), f"OUTPUT_PATH not found: {OUTPUT_PATH}"

train_annotation_files = glob(str(TRAIN_ANNOTATIONS_DIR / "*.xml"))
test_annotation_files  = glob(str(TEST_ANNOTATIONS_DIR / "*.xml"))

if not train_annotation_files:
    raise RuntimeError(f"No training annotation files found in {TRAIN_ANNOTATIONS_DIR}")
if not test_annotation_files:
    raise RuntimeError(f"No testing annotation files found in {TEST_ANNOTATIONS_DIR}")

# Target structure (inverse logic)
train_images_root = OUTPUT_PATH / "train" / "images"
test_images_root  = OUTPUT_PATH / "test" / "images"
train_images_root.mkdir(parents=True, exist_ok=True)
test_images_root.mkdir(parents=True, exist_ok=True)

def move_sequence(seq_name: str, dest_root: Path):
    src = ALL_IMAGES_DIR / seq_name
    dst = dest_root / seq_name

    if not src.is_dir():
        print(f"Warning: source sequence missing, skipped: {src}")
        return

    if dst.exists():
        print(f"Warning: destination already exists, skipped: {dst}")
        return

    shutil.move(str(src), str(dst))

# Move TRAIN sequences
for ann in tqdm(train_annotation_files, desc="Moving TRAIN sequences"):
    seq = Path(ann).stem
    move_sequence(seq, train_images_root)

# Move TEST sequences
for ann in tqdm(test_annotation_files, desc="Moving TEST sequences"):
    seq = Path(ann).stem
    move_sequence(seq, test_images_root)

print("Dataset successfully split by moving sequences.")

Moving TRAIN sequences: 100%|██████████| 60/60 [00:00<00:00, 1116.49it/s]


Moving TEST sequences: 100%|██████████| 40/40 [00:00<00:00, 3262.72it/s]

Dataset successfully split by moving sequences.


## 3. Convert UE-DETRAC -> YOLo (annotations)

In [3]:
# Classes -> YOLO ids
CLASS_MAPPING = {
    "car": 0,
    "bus": 1,
    "van": 2,
}

def process_detrac_annotations(annotations_dir, dataset_root, dataset_type):
    """
    Creates YOLO labels under:
      data_processed/<train|test>/labels/<sequence_name>/imgXXXXX.txt

    Reads images from:
      data_processed/<train|test>/images/<sequence_name>/imgXXXXX.jpg

    Does NOT generate images_annotated.
    Does NOT copy images again.
    """
    annotations_dir = Path(annotations_dir)
    dataset_root = Path(dataset_root)

    images_root = dataset_root / "images"
    labels_root = dataset_root / "labels"

    assert annotations_dir.is_dir(), f"annotations_dir not found: {annotations_dir}"
    assert images_root.is_dir(), f"images root not found (run split first): {images_root}"
    labels_root.mkdir(parents=True, exist_ok=True)

    processed_count = 0

    for xml_file in tqdm(list(annotations_dir.glob("*.xml")), desc=f"Processing {dataset_type} sequences"):
        sequence_name = xml_file.stem
        image_dir = images_root / sequence_name

        if not image_dir.is_dir():
            print(f"Warning: missing sequence folder, skipped: {image_dir}")
            continue

        # Read image size once per sequence
        try:
            first_jpg = sorted([p for p in image_dir.iterdir() if p.suffix.lower() == ".jpg"])[0]
            with Image.open(first_jpg) as img:
                img_width, img_height = img.size
        except Exception:
            print(f"Warning: could not read image size for {sequence_name}, skipped.")
            continue

        seq_labels_dir = labels_root / sequence_name
        seq_labels_dir.mkdir(parents=True, exist_ok=True)

        tree = ET.parse(str(xml_file))
        root = tree.getroot()

        for frame in root.findall("frame"):
            frame_num = int(frame.get("num"))
            image_filename = f"img{frame_num:05d}.jpg"
            label_filename = f"img{frame_num:05d}.txt"

            source_image_path = image_dir / image_filename
            dest_label_path = seq_labels_dir / label_filename

            if not source_image_path.exists():
                continue

            # Skip if label already generated
            if dest_label_path.exists():
                processed_count += 1
                continue

            yolo_annotations = []
            target_list = frame.find("target_list")
            if target_list is not None:
                for target in target_list.findall("target"):
                    box = target.find("box")
                    attribute = target.find("attribute")
                    if box is None or attribute is None:
                        continue

                    vehicle_type = attribute.get("vehicle_type")
                    if vehicle_type not in CLASS_MAPPING:
                        continue

                    class_id = CLASS_MAPPING[vehicle_type]

                    xmin = float(box.get("left"))
                    ymin = float(box.get("top"))
                    width = float(box.get("width"))
                    height = float(box.get("height"))

                    x_center = (xmin + width / 2) / img_width
                    y_center = (ymin + height / 2) / img_height
                    w_norm = width / img_width
                    h_norm = height / img_height

                    yolo_annotations.append(
                        f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"
                    )

            with open(dest_label_path, "w") as f:
                f.write("\n".join(yolo_annotations))

            processed_count += 1

    print(f"\n{dataset_type.capitalize()} - Total labels generated/skipped: {processed_count}")
    return processed_count


# Roots produced by the MOVE split code
train_dataset_root = OUTPUT_PATH / "train"
test_dataset_root  = OUTPUT_PATH / "test"

train_count = process_detrac_annotations(TRAIN_ANNOTATIONS_DIR, train_dataset_root, "train")
test_count  = process_detrac_annotations(TEST_ANNOTATIONS_DIR,  test_dataset_root,  "test")

print("\nConversion complete!")
print(f"Training: {train_count} labels")
print(f"Testing: {test_count} labels")

Processing train sequences: 100%|██████████| 60/60 [00:51<00:00,  1.17it/s]



Train - Total labels generated/skipped: 82082


Processing test sequences: 100%|██████████| 40/40 [00:28<00:00,  1.41it/s]


Test - Total labels generated/skipped: 56167

Conversion complete!
Training: 82082 labels
Testing: 56167 labels
